# Real use cases, measured — and what this is *not*

Four things people actually do with a topic store, each with a number attached, followed by an
honest map of what is missing compared to a full RAG stack, an ontology, and a vector database.

1. **Grounded answers**: does retrieved context make a small local model more accurate? (scored)
2. **Meaning, not keywords**: paraphrased questions over meeting notes.
3. **Languages**: where the local embedder is strong and where it is not.
4. **Session memory**: an agent stashing findings and getting them back later.

Needs the **Python 3 (trading)** kernel and Ollama with `nomic-embed-text` and `qwen2.5:7b-instruct`.

In [1]:
import sys, time, re
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import httpx
from slim_llm_memory import library

db = library(ROOT / ".usecases_nb")
RUN_LLM = True          # section 1 makes 8 short LLM calls (1–2 min each on a loaded CPU)

def llm(user, system="Answer in one or two sentences."):
    r = httpx.post("http://localhost:11434/api/chat", timeout=900, json={
        "model": LLM, "stream": False, "options": {"num_predict": 80, "temperature": 0},
        "messages": [{"role": "system", "content": system}, {"role": "user", "content": user}]})
    r.raise_for_status()
    return r.json()["message"]["content"].strip()

## 1. Grounded answers: with and without retrieval

The topic is this library's own docs. Four questions have short, checkable answers. Two things get measured:

- **Retrieval**: at which rank does the chunk that actually contains the answer appear?
- **Answering**: does a local model get it right cold, and grounded on the retrieved chunks?

In [2]:
docs = db.topic("slim-llm-memory")
docs.add([ROOT / "README.md", ROOT / "docs" / "IMPLEMENTATION.md"])

# (expected term, question as a user might type it, same question without the product name)
QA = [
    ("manifest", "In slim-llm-memory, which file is the atomic commit point of a flush?",
                 "Which file is the atomic commit point when flushing the index to disk?"),
    ("20%",      "In slim-llm-memory, at what tombstone ratio does compaction happen?",
                 "When are tombstoned items compacted away — at what percentage?"),
    ("nomic",    "Which local embedding model does slim-llm-memory use by default?",
                 "Which Ollama embedding model is the default?"),
    ("lock",     "How does slim-llm-memory stop two processes from writing the same index?",
                 "What prevents a second writer process from opening the same index directory?"),
]

def answer_rank(topic, q, must, k=10):
    hits = topic.ask(q, k=k, min_score=0.0).hits
    return next((i + 1 for i, h in enumerate(hits) if must.lower() in h.text.lower()), None)

print(f"{'expects':<9} {'rank, with product name':>24} {'rank, rephrased':>16}")
for must, q_named, q_plain in QA:
    print(f"{must:<9} {str(answer_rank(docs, q_named, must)):>24} {str(answer_rank(docs, q_plain, must)):>16}")

expects    rank, with product name  rank, rephrased


manifest                         6                1


20%                           None                1
nomic                            3                1


lock                             8                5


**What just happened.** With the product name in the question, the answer chunk ranks 3rd to 8th or is missing:
the rare token *slim-llm-memory* dominates the embedding and pulls the two "about this library" intro chunks to the
top. Ask the same thing in plain words and the answer chunk is rank 1 for three of four. The embedder matches
*meaning*; it has no notion of "this proper noun is just scoping". This is precisely the gap that hybrid
BM25 + dense retrieval and a reranker close (see the table at the end).

Now the answering half, with the plain questions, `k=6`, and a 7B model that follows context (the 3B one mostly
ignores it). Eight short calls; slow on CPU.

In [3]:
LLM = "qwen2.5:7b-instruct"
if RUN_LLM:
    rows = []
    for must, _, q in QA:
        cold = llm(q)
        r = docs.ask(q, k=6, min_score=0.0)
        warm = llm(f"{r.context}\n\nQuestion: {q}",
                   system="Answer in one or two sentences, strictly from the context. Cite [n].")
        rows.append((q, must, must.lower() in cold.lower(), must.lower() in warm.lower(), cold, warm))
    print(f"{'question':<66} {'cold':>5} {'grounded':>9}")
    for q, must, c, w, *_ in rows:
        print(f"{q[:64]:<66} {'✓' if c else '✗':>5} {'✓' if w else '✗':>9}   (expects '{must}')")
    print(f"\nscore: cold {sum(r[2] for r in rows)}/{len(rows)}   grounded {sum(r[3] for r in rows)}/{len(rows)}")

question                                                            cold  grounded
Which file is the atomic commit point when flushing the index to       ✗         ✓   (expects 'manifest')
When are tombstoned items compacted away — at what percentage?         ✗         ✓   (expects '20%')
Which Ollama embedding model is the default?                           ✗         ✓   (expects 'nomic')
What prevents a second writer process from opening the same inde       ✓         ✓   (expects 'lock')

score: cold 1/4   grounded 4/4


In [4]:
if RUN_LLM:
    for q, must, c, w, cold, warm in rows[:2]:
        print("Q:       ", q)
        print("COLD:    ", cold[:220])
        print("GROUNDED:", warm[:220], "\n")

Q:        Which file is the atomic commit point when flushing the index to disk?
COLD:     The atomic commit point when flushing the index to disk is typically associated with the write operations that update the transaction log (e.g., WAL files in PostgreSQL) or the journaling mechanism used by the database s
GROUNDED: The atomic commit point when flushing the index to disk is the `manifest.json` file, as mentioned in the Persistence model section of the context [n]. 

Q:        When are tombstoned items compacted away — at what percentage?
COLD:     Tombstoned items in a landfill are typically compacted when the site reaches about 60-80% full, though this can vary based on local regulations and operational practices.
GROUNDED: Tombstoned items are compacted when more than 20% of the items in `items.jsonl` are tombstoned. 



## 2. Meaning, not keywords

Six short notes from a fictional team. The questions share **no words** with the notes they should find.
`grep` would return nothing; the store returns the right note with a comfortable margin over the runner-up.

In [5]:
notes = db.topic("team notes")
notes.add({
    "release.md": "We agreed to ship the release on the first Monday of each month, right after the on-call handover.",
    "db.md":      "Postgres connection pool exhausted last night; raise max_connections and add pgbouncer.",
    "onboard.md": "New hires get laptop, VPN and repo access on day one; buddy assigned on day two.",
    "budget.md":  "Cloud spend is up 30% quarter over quarter, mostly egress; reviewing CDN options.",
    "hiring.md":  "Two backend openings approved; interviews start next sprint.",
    "retro.md":   "Retro outcome: fewer meetings, one written weekly update instead of standups.",
})

for q in ["rollout cadence", "why did the database fall over?", "first day checklist for a new colleague",
          "are we spending too much on infrastructure?", "did we decide anything about standups?"]:
    r = notes.ask(q, k=2, min_score=0.0)
    print(f"{q:<48} → {r.top.meta['doc']:<12} {r.top.score:.2f}   (next: {r.hits[1].meta['doc']} {r.hits[1].score:.2f})")

rollout cadence                                  → release.md   0.48   (next: budget.md 0.46)


why did the database fall over?                  → db.md        0.56   (next: budget.md 0.50)


first day checklist for a new colleague          → onboard.md   0.69   (next: release.md 0.49)


are we spending too much on infrastructure?      → budget.md    0.53   (next: db.md 0.41)


did we decide anything about standups?           → retro.md     0.64   (next: hiring.md 0.47)


## 3. Languages: strong within a language, weak across

`nomic-embed-text` is an English-first model. German notes queried **in German** work; queried **in English**
they do not reliably. This is the embedder, not the store: swap in a multilingual model (`bge-m3`, or a
cloud embedder) and the same code works across languages.

In [6]:
de = db.topic("privat")
de.add({
    "einkauf.md":  "Einkaufsliste: Milch, Brot, Eier, Butter und Kaffee.",
    "zahnarzt.md": "Zahnarzttermin am Dienstag um 9 Uhr, bitte nicht vergessen.",
    "steuer.md":   "Die Steuererklärung muss bis Ende Juli eingereicht werden.",
})
for q in ["Wann ist der Zahnarzt?", "Was muss ich einkaufen?", "Frist für die Steuer?",
          "when is the dentist appointment?", "what do I need from the supermarket?", "tax filing deadline"]:
    r = de.ask(q, k=1, min_score=0.0)
    print(f"{q:<40} → {r.top.meta['doc']:<12} {r.top.score:.2f}")

Wann ist der Zahnarzt?                   → zahnarzt.md  0.76


Was muss ich einkaufen?                  → einkauf.md   0.72
Frist für die Steuer?                    → steuer.md    0.73
when is the dentist appointment?         → steuer.md    0.41
what do I need from the supermarket?     → einkauf.md   0.48


tax filing deadline                      → steuer.md    0.45


## 4. Session memory for an agent

An agent working on a task stashes what it learns as plain text. Later prompts, phrased differently,
get those findings back — the "informed skill" loop. Adding a note costs one embedding; nothing to schema.

In [7]:
session = db.topic("session 2026-09-03")
session.add("Finding: the flaky test was caused by a shared tmp dir; fixed by using tmp_path per test.", name="f1")
session.add("Decision: keep numpy linear scan until the index passes 100k chunks, then evaluate faiss.", name="f2")
session.add("Blocker: Ollama on CPU embeds ~1 s per chunk under load; batch requests to 16 texts.", name="f3")

for q in ["why were tests flaky?", "when do we switch to an ANN index?", "how slow is embedding here?"]:
    r = session.ask(q, k=1, min_score=0.0)
    print(f"{q:<36} → {r.top.text}")

why were tests flaky?                → Finding: the flaky test was caused by a shared tmp dir; fixed by using tmp_path per test.
when do we switch to an ANN index?   → Decision: keep numpy linear scan until the index passes 100k chunks, then evaluate faiss.
how slow is embedding here?          → Blocker: Ollama on CPU embeds ~1 s per chunk under load; batch requests to 16 texts.


In [8]:
# ...and the whole library answers across topics, labelled, with one embedding call:
db.ask("what did we decide about the linear scan?", k=3)

ask('what did we decide about the linear scan?')  3 hit(s) · embed 49 ms · scan 0.24 ms
   1  0.68  session 2026-09-03/f2#0  Decision: keep numpy linear scan until the index passes 100k chunks, t
   2  0.57  slim-llm-memory/README.md#5 | Items | Pure-Python cosine | numpy linear scan (this lib) | faiss HN
   3  0.55  slim-llm-memory/IMPLEMENTATION.md#13 Acceptance: - No global state — everything hangs off the `Memory` / `T

## What this is — and what it is not

**What you have:** per-topic numpy stores, content-hashed incremental updates, cosine top-k in well under a
millisecond per topic, centroid routing across topics, a context block for any LLM, and atomic persistence.
About 1,800 lines of library code, numpy + httpx.

### Compared to a full RAG stack

| RAG component | Here | Missing / how to add |
|---|---|---|
| Chunking | paragraph packer, ~120 words | no overlap, no structure-aware splitting (headings, code blocks) |
| Embedding | one local/cloud embedder per store | no multilingual default; no per-query instruction prefixes |
| Retrieval | dense cosine top-k | **no hybrid BM25 + dense** (section 1's proper-noun failure), no MMR diversity, no metadata filters beyond `kind` |
| Reranking | none | a cross-encoder over the top-20 is the single biggest accuracy lever |
| Query handling | verbatim prompt | no rewriting (section 1 shows why it matters), no HyDE, no multi-query fusion |
| Generation | `answer()` = context + one Ollama call | no streaming, no citation checking, no refusal policy |
| Evaluation | section 1 above, by hand | no eval harness (recall@k, faithfulness) to tune k / chunking against |
| Conversation | stateless per call | no chat history, no summarised long-term memory of the dialogue |

### Compared to an ontology / knowledge graph

Retrieval here is by **similarity**, not by **structure**. There are no entities, no typed relations,
no schema, no inference. "Which services depend on Postgres?" can only be answered if a chunk happens to say so;
a graph would traverse `depends_on` edges. Two things are already in place to build on: the Obsidian parser keeps
`[[links]]` in `meta.links`, and `Memory.neighbours()` gives a cheap similarity edge. The planned graph layer
(`docs/IMPLEMENTATION.md` §8) adds typed edges over the same ids and a hybrid `related()` (0.6·cosine + 0.4·graph).
Entity and relation extraction would come from a local LLM pass at ingest — the spec's deferred "enrichment hook".

### Compared to a vector database

Single writer per store, linear scan (no ANN), no filtering language, no multi-tenancy, no replication.
By design: at < 100k chunks per topic none of that is needed, and the migration path (faiss behind the same
`search()`, SQLite-vss for a second writer) is documented in the README.

### When this is enough

One user or one agent, tens of topics, thousands to tens of thousands of chunks each, questions whose answers
live in a paragraph somewhere. That covers most personal and project knowledge. When questions become
relational ("everything that depends on X") or the corpus outgrows a laptop, graduate one component at a time.

In [9]:
db.close()